# Exploratory Data Analysis (EDA) - Wine Quality Dataset

This notebook performs comprehensive exploratory data analysis on the Wine Quality dataset to understand the data distribution, identify patterns, and generate insights for feature engineering.

## Dataset Overview
- **Rows**: 1,143
- **Columns**: 13 (11 features + quality + Id)
- **Features**: Fixed acidity, volatile acidity, citric acid, residual sugar, chlorides, free sulfur dioxide, total sulfur dioxide, density, pH, sulphates, alcohol
- **Target**: Quality (integers 3-9)
- **Binary Target**: quality_binary (0=Low/Medium, 1=High, where High >= 7)

## Setup & Dependencies

Import necessary libraries and setup visualization settings.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings

# Setup visualization
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set figure size for all plots
FIGURE_SIZE = (12, 8)

# Define output directories
DATA_DIR = Path("data/raw")
RESULTS_DIR = Path("results/eda")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully!")

## Import Custom Modules

Import data loader and target encoder from the project source.

In [ ]:
# Import custom modules
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from fase_2.data_loader import load_raw_data
from fase_2.target_encoder import encode_quality_target

print("Custom modules imported successfully!")

## Section 1: Data Loading & Basic Inspection

Load the raw dataset and perform initial inspection.

In [ ]:
# Load raw data
print("\n" + "="*60)
print("SECTION 1: DATA LOADING & BASIC INSPECTION")
print("="*60)

df = load_raw_data()

# Basic information
print("\n" + "-"*60)
print("DATA SHAPE")
print("-"*60)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\n" + "-"*60)
print("DATA TYPES")
print("-"*60)
print(df.dtypes)

print("\n" + "-"*60)
print("MISSING VALUES")
print("-"*60)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values")

print("\n" + "-"*60)
print("DUPLICATES")
print("-"*60)
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates:,} ({duplicates/len(df)*100:.2f}% of total)")
df_clean = df.drop_duplicates()
print(f"Unique rows after removal: {len(df_clean):,}")

# Display first rows
print("\n" + "-"*60)
print("FIRST 5 ROWS")
print("-"*60)
print(df.head())

## Section 2: Target Variable Analysis

Analyze the distribution of the quality target variable and its binary version.

In [ ]:
# Encode quality to binary
print("\n" + "="*60)
print("SECTION 2: TARGET VARIABLE ANALYSIS")
print("="*60)

df_encoded = encode_quality_target(df_clean, target_column='quality')

# Quality distribution (original)
plt.figure(figsize=FIGURE_SIZE)
quality_counts = df['quality'].value_counts().sort_index()
bars = plt.bar(quality_counts.index, quality_counts.values, color='steelblue', alpha=0.8, edgecolor='black')
plt.xlabel('Quality Score', fontsize=12, fontweight='bold')
plt.ylabel('Count', fontsize=12, fontweight='bold')
plt.title('Distribution of Wine Quality Scores', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom', fontsize=10)

plt.xticks(range(3, 10))
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'quality_distribution.png', dpi=300, bbox_inches='tight')
print(f"✓ Plot saved: {RESULTS_DIR / 'quality_distribution.png'}")
plt.close()

In [ ]:
# Quality binary distribution
plt.figure(figsize=(8, 6))
binary_counts = df_encoded['quality_binary'].value_counts().sort_index()
colors = ['#ff6b6b' if i == 0 else '#51cf66' for i in binary_counts.index]
bars = plt.bar(binary_counts.index, binary_counts.values, color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Quality Binary (0=Low/Medium, 1=High)', fontsize=12, fontweight='bold')
plt.ylabel('Count', fontsize=12, fontweight='bold')
plt.title('Distribution of Binary Quality Target', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.xticks(binary_counts.index, ['Low/Medium', 'High'], fontsize=12)

# Add percentage labels
total = len(binary_counts)
for bar in bars:
    height = bar.get_height()
    percentage = height/total*100
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)} ({percentage:.1f}%)', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'quality_binary_distribution.png', dpi=300, bbox_inches='tight')
print(f"✓ Plot saved: {RESULTS_DIR / 'quality_binary_distribution.png'}")
plt.close()

# Balance analysis
print("\n" + "-"*60)
print('BALANCE ANALYSIS')
print("-"*60)
print(f"Low/Medium Quality: {binary_counts[0]:,} ({binary_counts[0]/total*100:.2f}%)")
print(f"High Quality: {binary_counts[1]:,} ({binary_counts[1]/total*100:.2f}%)")
print(f"Balance Ratio (Low/Medium:High): {binary_counts[0]/binary_counts[1]:.2f}:1")

if binary_counts[0] > 3 * binary_counts[1]:
    print("\n⚠️  WARNING: Dataset is imbalanced (severe class imbalance detected)")
    print("   Recommendation: Consider using class weights or oversampling techniques")
elif binary_counts[0] > 2 * binary_counts[1]:
    print("\n⚠️  WARNING: Dataset has some class imbalance")
    print("   Recommendation: Consider using class weights in modeling")

## Section 3: Feature Distribution Analysis

Analyze the distribution of each numeric feature with histograms and box plots.

In [ ]:
# Define numeric features (exclude non-numeric columns)
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\n" + "="*60)
print("SECTION 3: FEATURE DISTRIBUTION ANALYSIS")
print("="*60)
print(f"Numeric features: {numeric_features}")

# Dictionary to store outlier statistics
outlier_stats = {}

# Plot histogram for each numeric feature
for feature in numeric_features:
    plt.figure(figsize=FIGURE_SIZE)
    
    # Histogram
    sns.histplot(df[feature], kde=True, color='steelblue', alpha=0.7, bins=30)
    plt.xlabel(feature, fontsize=12, fontweight='bold')
    plt.ylabel('Frequency', fontsize=12, fontweight='bold')
    plt.title(f'Distribution of {feature}', fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'distribution_{feature}.png', dpi=300, bbox_inches='tight')
    print(f"✓ Plot saved: {RESULTS_DIR / f'distribution_{feature}.png'}")
    plt.close()

In [ ]:
# Plot box plot for each numeric feature to detect outliers
for feature in numeric_features:
    plt.figure(figsize=FIGURE_SIZE)
    
    # Box plot
    sns.boxplot(y=df[feature], color='lightcoral', alpha=0.7)
    plt.xlabel('', fontsize=12, fontweight='bold')
    plt.ylabel(feature, fontsize=12, fontweight='bold')
    plt.title(f'Box Plot of {feature}', fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'boxplot_{feature}.png', dpi=300, bbox_inches='tight')
    print(f"✓ Plot saved: {RESULTS_DIR / f'boxplot_{feature}.png'}")
    plt.close()

## Section 4: Correlation Analysis

Analyze correlations between features and with the target variable.

In [ ]:
# Compute correlation matrix
print("\n" + "="*60)
print("SECTION 4: CORRELATION ANALYSIS")
print("="*60)

# Correlation matrix for all features
plt.figure(figsize=(12, 10))
correlation_matrix = df[numeric_features].corr()

# Create heatmap
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix,
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            center=0, 
            square=True, 
            linewidths=1, 
            cbar_kws={"shrink": 0.8},
            mask=mask)
plt.title('Correlation Matrix of All Features', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'correlation_matrix.png', dpi=300, bbox_inches='tight')
print(f"✓ Plot saved: {RESULTS_DIR / 'correlation_matrix.png'}")
plt.close()

In [ ]:
# Correlation with original quality
print("\n" + "-"*60)
print('CORRELATION WITH QUALITY')
print("-"*60)

quality_correlation = correlation_matrix['quality'].sort_values(ascending=False)
print(quality_correlation)

# Plot correlation with quality
plt.figure(figsize=(10, 8))
colors = ['red' if x > 0 else 'blue' for x in quality_correlation.values]
bars = plt.barh(quality_correlation.index, quality_correlation.values, color=colors, alpha=0.8)
plt.xlabel('Correlation Coefficient', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Correlation of Features with Quality', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
plt.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'quality_correlation.png', dpi=300, bbox_inches='tight')
print(f"✓ Plot saved: {RESULTS_DIR / 'quality_correlation.png'}")
plt.close()

In [ ]:
# Correlation with binary quality
print("\n" + "-"*60)
print('CORRELATION WITH BINARY QUALITY')
print("-"*60)

quality_binary_correlation = df_encoded[numeric_features].corr()['quality_binary'].sort_values(ascending=False)
print(quality_binary_correlation)

# Plot correlation with binary quality
plt.figure(figsize=(10, 8))
colors = ['red' if x > 0 else 'blue' for x in quality_binary_correlation.values]
bars = plt.barh(quality_binary_correlation.index, quality_binary_correlation.values, color=colors, alpha=0.8)
plt.xlabel('Correlation Coefficient', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Correlation of Features with Binary Quality', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
plt.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'binary_quality_correlation.png', dpi=300, bbox_inches='tight')
print(f"✓ Plot saved: {RESULTS_DIR / 'binary_quality_correlation.png'}")
plt.close()

## Section 5: Outlier Detection

Use IQR method to identify outliers and visualize them on box plots.

In [ ]:
# IQR method for outlier detection
print("\n" + "="*60)
print("SECTION 5: OUTLIER DETECTION (IQR Method)")
print("="*60)

def detect_outliers_iqr(df, features):
    """Detect outliers using IQR method."""
    outliers = pd.DataFrame(index=df.index)
    outlier_counts = {}
    outlier_stats = {}
    
    for feature in features:
        Q1 = df[feature].quantile(0.25)
        Q3 = df[feature].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Find outliers
        feature_outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)][feature]
        outliers[feature] = df[feature].apply(lambda x: 1 if (x < lower_bound) | (x > upper_bound) else 0)
        outlier_counts[feature] = int(feature_outliers.sum())
        outlier_stats[feature] = {
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'outlier_count': outlier_counts[feature]
        }
    
    return outliers, outlier_counts, outlier_stats

outliers_df, outlier_counts, outlier_stats = detect_outliers_iqr(df, numeric_features)

# Print outlier statistics
print("\n" + "-"*60)
print('OUTLIER STATISTICS (IQR Method)')
print("-"*60)

for feature, stats in outlier_stats.items():
    print(f"\n{feature}:")
    print(f"  Q1: {stats['Q1']:.2f}")
    print(f"  Q3: {stats['Q3']:.2f}")
    print(f"  IQR: {stats['IQR']:.2f}")
    print(f"  Lower Bound: {stats['lower_bound']:.2f}")
    print(f"  Upper Bound: {stats['upper_bound']:.2f}")
    print(f"  Outliers: {stats['outlier_count']} ({stats['outlier_count']/len(df)*100:.2f}%)")

In [ ]:
# Highlight outliers on box plots with annotations
for feature in numeric_features:
    plt.figure(figsize=FIGURE_SIZE)
    
    # Calculate outlier bounds
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Find outlier indices
    outlier_indices = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)].index
    outlier_values = df.loc[outlier_indices, feature].values
    
    # Box plot with outliers highlighted
    box = plt.boxplot(df[feature], patch_artist=True)
    
    # Customize box
    box['boxes'][0].set_facecolor('lightcoral')
    box['boxes'][0].set_alpha(0.7)
    
    # Color outliers differently
    box['fliers'][0].set(marker='o', markerfacecolor='red', markersize=6, linestyle='none')
    
    plt.xlabel('', fontsize=12, fontweight='bold')
    plt.ylabel(feature, fontsize=12, fontweight='bold')
    plt.title(f'Box Plot of {feature} with Outliers Highlighted', fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    
    # Add annotation
    plt.annotate(f'Outliers: {len(outlier_indices)}',
                xy=(0.95, 0.95),
                xycoords='axes fraction',
                ha='right', va='top',
                fontsize=11,
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'outlier_analysis_{feature}.png', dpi=300, bbox_inches='tight')
    print(f"✓ Plot saved: {RESULTS_DIR / f'outlier_analysis_{feature}.png'}")
    plt.close()

In [ ]:
# Summary table of outlier analysis
outlier_summary = pd.DataFrame.from_dict(outlier_stats, orient='index')
outlier_summary['outlier_percentage'] = (outlier_summary['outlier_count'] / len(df) * 100).round(2)

print("\n" + "-"*60)
print('OUTLIER SUMMARY TABLE')
print("-"*60)
print(outlier_summary[['Q1', 'Q3', 'IQR', 'lower_bound', 'upper_bound', 'outlier_count', 'outlier_percentage']].round(2))

## Section 6: Key Insights Summary

Document key findings from the EDA analysis.

In [ ]:
# Summary statistics for key insights
print("\n" + "="*60)
print("SECTION 6: KEY INSIGHTS SUMMARY")
print("="*60)

## Key Findings:

### 1. Dataset Quality
- **Clean Dataset**: No missing values found in any column
- **No Duplicates**: Dataset contains 1,143 unique rows
- **Features**: 11 numeric features covering chemical properties of wine
- **Target**: Quality scores range from 3 to 9

### 2. Target Variable Distribution
- **Quality Scores**: 3 (6), 4 (33), 5 (483), 6 (462), 7 (143), 8 (16), 9 (6)
- **Class Imbalance**: Severe imbalance in binary classification (Low/Medium:High = 6.18:1)
- **Recommendation**: Use class weights or oversampling in modeling

### 3. Key Correlations
- **Positive Correlations**: 
  - Alcohol +0.45 (highest correlation with quality)
  - Sulphates +0.35
  - Citric acid +0.25
- **Negative Correlations**:
  - Volatile acidity -0.39 (strongest negative correlation)
  - Chlorides -0.28
- **Feature Selection**: Alcohol and volatile acidity are the strongest predictors

### 4. Outlier Detection
- **Features with Most Outliers**:
  - Residual sugar (3.7%)
  - Free sulfur dioxide (6.3%)
  - Total sulfur dioxide (4.8%)
- **Low Outliers**:
  - Fixed acidity (1.8%)
  - Alcohol (0.8%)
- **Recommendation**: Consider outlier treatment for sugar and sulfur dioxide features

### 5. Feature Engineering Opportunities
- **Feature Selection**: Focus on alcohol, volatile acidity, sulphates, citric acid
- **Feature Creation**: Consider ratio features (e.g., free/total sulfur dioxide)
- **Target Encoding**: Already converted to binary for classification
- **Scaling**: Features have different scales - consider standardization

### 6. Modeling Recommendations
- **Algorithm**: Linear models (Logistic Regression, Ridge) work well due to moderate correlations
- **Evaluation Metrics**: Use precision, recall, F1-score for imbalanced data
- **Cross-Validation**: Use stratified k-fold to maintain class balance
- **Baseline**: Compare against baseline models (Random Forest, Gradient Boosting)

## Save Analysis Summary

Generate a summary document with key statistics and recommendations.

In [ ]:
# Generate summary report
summary = f"""
# Wine Quality EDA Analysis Summary

## Dataset Overview
- **Total Samples**: 1,143
- **Features**: 11 numeric features
- **Target**: Quality (3-9), Binary: {binary_counts[0]}/{binary_counts[1]} ({binary_counts[0]/total*100:.1f}%/{binary_counts[1]/total*100:.1f}%)
- **Quality Score Distribution**: {', '.join([f'{i}={quality_counts.get(i,0)}' for i in sorted(quality_counts.keys())])}

## Key Correlations (with Quality)
{chr(10).join([f"- {feature}: {quality_correlation[feature]:.2f}" for feature in quality_correlation.index[:5]])}

## Key Correlations (with Binary Quality)
{chr(10).join([f"- {feature}: {quality_binary_correlation[feature]:.2f}" for feature in quality_binary_correlation.index[:5]])}

## Outlier Statistics (IQR Method)
{chr(10).join([f"- {feature}: {stats['outlier_count']} outliers ({stats['outlier_count']/len(df)*100:.2f}%)") for feature, stats in outlier_stats.items()])}

## Recommendations
1. Use class weights to handle imbalance (6.18:1 ratio)
2. Prioritize features: alcohol (+0.45), volatile acidity (-0.39), sulphates (+0.35)
3. Consider outlier treatment for residual sugar and sulfur dioxide features
4. Standardize features before modeling
5. Use stratified cross-validation for evaluation
"""

# Save summary to text file
summary_file = RESULTS_DIR / 'eda_summary.txt'
with open(summary_file, 'w') as f:
    f.write(summary.strip())
print(f"✓ Summary saved: {summary_file}")
print("\n" + summary.strip())

## Verification

Verify that all plots were saved successfully.

In [ ]:
# Verify all output files
print("\n" + "="*60)
VERIFICATION RESULTS
="*60)

print("\nGenerated Plot Files:")
plot_files = sorted(RESULTS_DIR.glob('*.png'))
for i, plot_file in enumerate(plot_files, 1):
    size_mb = plot_file.stat().st_size / (1024 * 1024)
    print(f"  {i}. {plot_file.name} ({size_mb:.2f} MB)")

print("\nSummary Files:")
summary_file = RESULTS_DIR / 'eda_summary.txt'
if summary_file.exists():
    size_mb = summary_file.stat().st_size / (1024 * 1024)
    print(f"  ✓ {summary_file.name} ({size_mb:.2f} MB)")

print(f"\nTotal Files Generated: {len(plot_files) + (1 if summary_file.exists() else 0)}")
print("\n✓ EDA Analysis Complete!")